In [1]:
import sys
import os
import json
from dotenv import load_dotenv
import mongoengine

load_dotenv()

mongoengine.connect(
    db=os.getenv("MONGO_DB_NAME"),
    host=os.getenv("MONGO_URI")
)

sys.path.append(os.path.abspath(".."))

from products.models import Product
from products.category_model import ProductCategory

print("✅ MongoDB Connected!")
print(f"📦 Total products: {Product.objects.count()}")

✅ MongoDB Connected!
📦 Total products: 95


In [2]:
def get_product_info(product_name: str) -> dict:
    """
    Tool 1: Get product details from MongoDB by name
    Returns name, brand, category, price
    """
    # Search by name (case insensitive)
    product = Product.objects(name__icontains=product_name).first()

    if not product:
        return {"error": f"Product '{product_name}' not found in inventory"}

    try:
        category = product.category.title if product.category else "Unknown"
    except:
        category = "Unknown"

    return {
        "product_id": str(product.id),
        "name": product.name,
        "brand": product.brand,
        "category": category,
        "price": float(str(product.price)),
        "description": product.description
    }

# Test it
print("🔧 Testing get_product_info()")
print("=" * 50)

test_products = ["iPhone 14", "Coffee Maker", "Jeans"]
for name in test_products:
    result = get_product_info(name)
    print(f"\n Product: '{name}'")
    print(f"  Result: {result}")

🔧 Testing get_product_info()

 Product: 'iPhone 14'
  Result: {'product_id': '69d689bfe2f4e4b501f76643', 'name': 'iPhone 14', 'brand': 'Apple', 'category': 'Electronics', 'price': 80000.0, 'description': 'Latest Apple smartphone'}

 Product: 'Coffee Maker'
  Result: {'product_id': '69d689bfe2f4e4b501f76642', 'name': 'Coffee Maker', 'brand': 'Philips', 'category': 'Kitchen Essentials', 'price': 3500.0, 'description': 'Drip coffee maker'}

 Product: 'Jeans'
  Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'brand': 'Levis', 'category': 'Clothing', 'price': 2000.0, 'description': 'Denim slim fit jeans'}


In [3]:
def check_inventory(product_name: str) -> dict:
    """
    Tool 2: Check stock levels for a product
    Returns stock quantity and availability status
    """
    product = Product.objects(name__icontains=product_name).first()

    if not product:
        return {"error": f"Product '{product_name}' not found"}

    stock = product.quantity_in_warehouse

    if stock == 0:
        status = "OUT OF STOCK"
    elif stock < 10:
        status = "LOW STOCK"
    elif stock < 50:
        status = "IN STOCK"
    else:
        status = "WELL STOCKED"

    return {
        "product_id": str(product.id),
        "name": product.name,
        "stock": stock,
        "status": status,
        "can_fulfill": stock > 0
    }

# Test it
print("🔧 Testing check_inventory()")
print("=" * 50)

test_products = ["iPhone 14", "Samsung 4K TV", "Jeans"]
for name in test_products:
    result = check_inventory(name)
    print(f"\n Product: '{name}'")
    print(f"  Result: {result}")

🔧 Testing check_inventory()

 Product: 'iPhone 14'
  Result: {'product_id': '69d689bfe2f4e4b501f76643', 'name': 'iPhone 14', 'stock': 0, 'status': 'OUT OF STOCK', 'can_fulfill': False}

 Product: 'Samsung 4K TV'
  Result: {'product_id': '69d807bcd9dbf744053d4327', 'name': 'Samsung 4K TV 55 inch', 'stock': 45, 'status': 'IN STOCK', 'can_fulfill': True}

 Product: 'Jeans'
  Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'stock': 0, 'status': 'OUT OF STOCK', 'can_fulfill': False}


In [5]:
def calculate_quote(product_name: str, quantity: int) -> dict:
    """
    Tool 3: Calculate quote with bulk discount rules
    Discount rules:
      - Quantity 1-10   → 0% discount
      - Quantity 11-50  → 10% discount
      - Quantity 51+    → 20% discount (MAX allowed)
    """
    # Get product info
    product = Product.objects(name__icontains=product_name).first()

    if not product:
        return {"error": f"Product '{product_name}' not found"}

    # Check stock
    stock = product.quantity_in_warehouse
    if stock < quantity:
        return {
            "error": f"Insufficient stock. Requested: {quantity}, Available: {stock}"
        }

    unit_price = float(str(product.price))

    # Apply discount rules
    if quantity <= 10:
        discount_pct = 0
        discount_reason = "No discount for small orders"
    elif quantity <= 50:
        discount_pct = 10
        discount_reason = "10% bulk discount for orders of 11-50 units"
    else:
        discount_pct = 20
        discount_reason = "20% bulk discount for orders of 51+ units"

    # Calculate prices
    original_total = unit_price * quantity
    discount_amount = original_total * (discount_pct / 100)
    final_total = original_total - discount_amount

    return {
        "product_name": product.name,
        "brand": product.brand,
        "quantity": quantity,
        "unit_price": round(unit_price, 2),
        "discount_percentage": discount_pct,
        "discount_reason": discount_reason,
        "original_total": round(original_total, 2),
        "discount_amount": round(discount_amount, 2),
        "final_total": round(final_total, 2),
        "currency": "INR"
    }

# Test it
print("🔧 Testing calculate_quote()")
print("=" * 50)

test_cases = [
    ("iPhone 14", 5),    # no discount
    ("Air Conditioner", 20),   # 10% discount
    ("Black+Decker Blender", 60),   # 20% discount
    ("Nestle Chocolate Chips", 100),      # 20% discount
]

for name, qty in test_cases:
    result = calculate_quote(name, qty)
    print(f"\n Product: '{name}' | Quantity: {qty}")
    print(f"  {result}")

🔧 Testing calculate_quote()

 Product: 'iPhone 14' | Quantity: 5
  {'error': 'Insufficient stock. Requested: 5, Available: 0'}

 Product: 'Air Conditioner' | Quantity: 20
  {'product_name': 'Air Conditioner', 'brand': 'LG', 'quantity': 20, 'unit_price': 129.99, 'discount_percentage': 10, 'discount_reason': '10% bulk discount for orders of 11-50 units', 'original_total': 2599.8, 'discount_amount': 259.98, 'final_total': 2339.82, 'currency': 'INR'}

 Product: 'Black+Decker Blender' | Quantity: 60
  {'error': 'Insufficient stock. Requested: 60, Available: 50'}

 Product: 'Nestle Chocolate Chips' | Quantity: 100
  {'product_name': 'Nestle Chocolate Chips', 'brand': 'Nestle', 'quantity': 100, 'unit_price': 1.99, 'discount_percentage': 20, 'discount_reason': '20% bulk discount for orders of 51+ units', 'original_total': 199.0, 'discount_amount': 39.8, 'final_total': 159.2, 'currency': 'INR'}


In [13]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Define tools for the Agent in OpenAI function calling format
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_product_info",
            "description": "Get product details like price, brand, and category from the inventory database",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The name or partial name of the product to look up"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check how many units of a product are currently in stock",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The name of the product to check inventory for"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_quote",
            "description": "Calculate the total price quote for a product with quantity-based bulk discounts applied",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "The name of the product"
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "The number of units requested"
                    }
                },
                "required": ["product_name", "quantity"]
            }
        }
    }
]

# Map tool names to actual functions
TOOL_MAP = {
    "get_product_info": get_product_info,
    "check_inventory": check_inventory,
    "calculate_quote": calculate_quote
}

print("✅ Agent tools defined!")
print(f"   Tools available: {list(TOOL_MAP.keys())}")

✅ Agent tools defined!
   Tools available: ['get_product_info', 'check_inventory', 'calculate_quote']


In [ ]:
def run_quote_agent(user_request: str) -> str:
    print(f"\n🤖 Agent received: '{user_request}'")
    print("=" * 60)

    messages = [
        {
            "role": "system",
            "content": """You are a Quote Agent for a product inventory system.
When a user asks for a product quote, you must:
1. Find the product using get_product_info
2. Check stock using check_inventory
3. Calculate the quote using calculate_quote
4. Return a clear, friendly quote summary

Always use all 3 tools in order before giving a final answer.
Be helpful and mention the discount if applicable.
Always use proper JSON format for function calls."""
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    step = 1
    max_retries = 3

    while True:
        print(f"\n⚙️  Step {step} — Calling LLM...")

        # Retry logic for malformed tool calls
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=messages,
                    tools=TOOLS,
                    tool_choice="auto"
                )
                break  # success — exit retry loop
            except Exception as e:
                if "tool_use_failed" in str(e) and attempt < max_retries - 1:
                    print(f"   ⚠️ Tool call failed (attempt {attempt+1}), retrying...")
                    continue
                else:
                    return f"Agent encountered an error: {str(e)}"

        message = response.choices[0].message

        # If no tool calls — agent is done
        if not message.tool_calls:
            print(f"\n✅ Agent final answer ready!")
            return message.content

        # Append raw message object
        messages.append(message)

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"   🔧 Calling tool: {tool_name}({tool_args})")

            tool_result = TOOL_MAP[tool_name](**tool_args)

            print(f"   📤 Result: {tool_result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

        step += 1

        if step > 10:
            return "Agent exceeded maximum steps"

print("✅ Quote Agent ready!")

In [32]:
# Test with different requests
test_requests = [
    "Can I get a quote for 60 Jeans?",
    "I want to order 25 Coffee Maker , is there a discount?",
]

for request in test_requests:
    print("\n" + "🔵" * 30)
    result = run_quote_agent(request)
    print(f"\n💬 Final Quote:\n{result}")
    print("🔵" * 30)


🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵🔵

🤖 Agent received: 'Can I get a quote for 60 Jeans?'

⚙️  Step 1 — Calling LLM...
   🔧 Calling tool: get_product_info({'product_name': 'Jeans'})
   📤 Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'brand': 'Levis', 'category': 'Clothing', 'price': 2000.0, 'description': 'Denim slim fit jeans'}
   🔧 Calling tool: check_inventory({'product_name': 'Jeans'})
   📤 Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'stock': 0, 'status': 'OUT OF STOCK', 'can_fulfill': False}
   🔧 Calling tool: calculate_quote({'product_name': 'Jeans', 'quantity': 60})
   📤 Result: {'error': 'Insufficient stock. Requested: 60, Available: 0'}

⚙️  Step 2 — Calling LLM...

✅ Agent final answer ready!

💬 Final Quote:
I've checked on the Jeans you're interested in. Unfortunately, the product is currently out of stock, and I won't be able to provide a quote for 60 units. If you'd like, I can try to find an alternative product or assist you with a

In [36]:
EVAL_CASES = [
    # Simple scenarios
    {
        "request": "Can I get 20 Jeans?",
        "expected_discount": 10,
        "expected_product": "Jeans",
        "scenario": "simple"
    },
    # Complex scenarios
    {
        "request": "I need 50 Coffee Maker for a hotel chain, can I get a deal?",
        "expected_discount": 10,
        "expected_product": "Coffee Maker",
        "scenario": "complex"
    },
    {
        "request": "We're buying 30 Samsung 4K TV 55 inch for our office, what's the best price?",
        "expected_discount": 10,
        "expected_product": "Samsung",
        "scenario": "complex"
    },
]

print("📊 Quote Agent Eval Suite")
print("=" * 60)

passed = 0
total = len(EVAL_CASES)

for case in EVAL_CASES:
    print(f"\n🔎 [{case['scenario'].upper()}] '{case['request']}'")

    # Run agent
    result = run_quote_agent(case['request'])

    # Check if expected product mentioned in result
    product_found = case['expected_product'].lower() in result.lower()

    # Check if correct discount mentioned
    discount_str = str(case['expected_discount'])
    discount_found = discount_str in result

    success = product_found and discount_found
    if success:
        passed += 1

    status = "✅ PASSED" if success else "❌ FAILED"
    print(f"\n   {status}")
    print(f"   Product found: {'✅' if product_found else '❌'}")
    print(f"   Discount ({case['expected_discount']}%) found: {'✅' if discount_found else '❌'}")

print("\n" + "=" * 60)
print(f"📈 Eval Results: {passed}/{total} passed ({passed/total*100:.1f}%)")

📊 Quote Agent Eval Suite

🔎 [SIMPLE] 'Can I get 20 Jeans?'

🤖 Agent received: 'Can I get 20 Jeans?'

⚙️  Step 1 — Calling LLM...
   🔧 Calling tool: get_product_info({'product_name': 'Jeans'})
   📤 Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'brand': 'Levis', 'category': 'Clothing', 'price': 2000.0, 'description': 'Denim slim fit jeans'}
   🔧 Calling tool: check_inventory({'product_name': 'Jeans'})
   📤 Result: {'product_id': '69d689bfe2f4e4b501f76646', 'name': 'Jeans', 'stock': 0, 'status': 'OUT OF STOCK', 'can_fulfill': False}
   🔧 Calling tool: calculate_quote({'product_name': 'Jeans', 'quantity': 20})
   📤 Result: {'error': 'Insufficient stock. Requested: 20, Available: 0'}

⚙️  Step 2 — Calling LLM...

✅ Agent final answer ready!

   ❌ FAILED
   Product found: ✅
   Discount (10%) found: ❌

🔎 [COMPLEX] 'I need 50 Coffee Maker for a hotel chain, can I get a deal?'

🤖 Agent received: 'I need 50 Coffee Maker for a hotel chain, can I get a deal?'

⚙️  Step 1 — Ca

In [41]:
def calculate_quote_with_policy(product_name: str, quantity: int,
                                 requested_discount: float = None) -> dict:
    """
    Advanced version of calculate_quote with Policy Check
    Ensures discount never exceeds 20% — even if AI requests more
    """
    MAX_DISCOUNT = 20.0  # Business rule — never exceed this!

    result = calculate_quote(product_name, quantity)

    if "error" in result:
        return result

    # Policy check — if AI somehow requested higher discount
    policy_warning = None
    if requested_discount and requested_discount > MAX_DISCOUNT:
        policy_warning = (
            f"⚠️ Policy Override: Requested discount {requested_discount}% "
            f"exceeds maximum allowed {MAX_DISCOUNT}%. "
            f"Discount capped at {MAX_DISCOUNT}%."
        )
        print(f"\n🚨 POLICY CHECK TRIGGERED: {policy_warning}")

        # Recalculate with max discount
        unit_price = result["unit_price"]
        quantity = result["quantity"]
        original_total = unit_price * quantity
        discount_amount = original_total * (MAX_DISCOUNT / 100)
        final_total = original_total - discount_amount

        result["discount_percentage"] = MAX_DISCOUNT
        result["discount_amount"] = round(discount_amount, 2)
        result["final_total"] = round(final_total, 2)
        result["policy_override"] = True
        result["policy_warning"] = policy_warning
    else:
        result["policy_override"] = False

    return result

# ── Test policy check ───────────────────────────────────────────
print("🔧 Testing Policy Check")
print("=" * 50)

print("\n✅ Normal request (20% discount — within policy):")
result = calculate_quote_with_policy("Coffee Maker", 50)
if "error" in result:
    print(f"  Error: {result['error']}")
else:
    print(f"  Discount: {result['discount_percentage']}% | Override: {result['policy_override']}")

print("\n🚨 Excessive discount request (35% — policy override!):")
result = calculate_quote_with_policy("Coffee Maker", 50, requested_discount=35)
if "error" in result:
    print(f"  Error: {result['error']}")
else:
    print(f"  Discount: {result['discount_percentage']}% | Override: {result['policy_override']}")
    if result.get("policy_warning"):
        print(f"  Warning: {result['policy_warning']}")

🔧 Testing Policy Check

✅ Normal request (20% discount — within policy):
  Discount: 10% | Override: False

🚨 Excessive discount request (35% — policy override!):

🚨 POLICY CHECK TRIGGERED: ⚠️ Policy Override: Requested discount 35% exceeds maximum allowed 20.0%. Discount capped at 20.0%.
  Discount: 20.0% | Override: True


In [40]:
# Debug — test calculate_quote first
print("Testing base function first...")
test = calculate_quote("Coffee Maker", 60)
print(f"Base result: {test}")

Testing base function first...
Base result: {'error': 'Insufficient stock. Requested: 60, Available: 55'}
